### Imports and CUDA

In [1]:
# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import json
import os
from helper_functions import *
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

/home/sengkiat/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [4]:
# Detect available device: CUDA (Google Colab GPU) > DirectML (AMD GPU) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_directml
        device = torch_directml.device()
        print(f"Using DirectML: {device}")
    except:
        device = torch.device("cpu")
        print(f"Using CPU")

Using CUDA: Tesla T4


### DvXray Dataset
A large-scale dual-view X-ray baggage dataset for prohibited item detection.

- **Views:** Overlook (OL) & Side (SD) X-ray images
- **15 threat classes:** Gun, Knife, Hammer, Battery, etc.
- **Negative samples:** Benign baggage
- **Annotations:** JSON with bounding boxes

In [5]:
# DvXray Dataset
class DvXrayDataset(Dataset):
    """DvXray Dataset: dual-view X-ray baggage dataset for prohibited item detection.
    
    Args:
        transform: Optional transform to apply to each image.
        download: If True, downloads the dataset from Google Drive if not found.
    """
    
    def __init__(self, transform=None, download=False):
        # Download if requested
        if download and not check_dvxray_exists():
            download_and_extract_dvxray()
        neg_dir, pos_dir = get_directories()
        
        # Read dataset from directories
        self.samples = []
        self.labels = {}  # cache: ol_path -> list of label strings
        self.transform = transform

        # Add negative images into dataset
        for fname in os.listdir(neg_dir):
            if fname.endswith('_OL.png'):
                base = fname.replace('_OL.png', '')
                ol = os.path.join(neg_dir, base + '_OL.png')
                sd = os.path.join(neg_dir, base + '_SD.png')
                if os.path.exists(sd):
                    self.samples.append((ol, sd, 0))    # negatives = 0
                    self.labels[ol] = ["Benign"]
                else:
                    print(f"Missing SD for negative: {base}")

        # Add positive images into dataset
        for fname in os.listdir(pos_dir):
            if fname.endswith('_OL.png'):
                base = fname.replace('_OL.png', '')
                ol = os.path.join(pos_dir, base + '_OL.png')
                sd = os.path.join(pos_dir, base + '_SD.png')
                if os.path.exists(sd):
                    self.samples.append((ol, sd, 1))    # positives = 1
                    json_path = os.path.join(pos_dir, base + '.json')
                    obj_labels = []
                    if os.path.exists(json_path):
                        with open(json_path) as f:
                            data = json.load(f)
                        objects = data.get("objects")
                        if isinstance(objects, list) and len(objects) > 0:
                            for obj in objects:
                                obj_labels.append(obj["label"])
                        else:
                            obj_labels = ["Benign"]
                    else:
                        obj_labels = ["Benign"]
                    self.labels[ol] = obj_labels
                else:
                    print(f"Missing SD for positive: {base}")

        # Build label_map: Benign=0, threat classes sorted from 1
        all_labels = set()
        for label_list in self.labels.values():
            all_labels.update(label_list)

        self.label_map = {"Benign": 0}
        self.label_map.update(
            {label: i + 1 for i, label in enumerate(sorted(all_labels - {"Benign"}))}
        )
        self.num_classes = len(self.label_map)
    
    def __getitem__(self, idx):
        """Returns (image, vector, binary)"""
        ol_path, sd_path, binary_label = self.samples[idx]
        ol = Image.open(ol_path).convert('RGB')
        sd = Image.open(sd_path).convert('RGB')
        if self.transform:
            ol = self.transform(ol)
            sd = self.transform(sd)

        image = torch.cat([ol, sd], dim=0)
        multi_hot = torch.zeros(self.num_classes)
        for label_str in self.labels[ol_path]:
            if label_str in self.label_map:
                multi_hot[self.label_map[label_str]] = 1.0
    
        # (6 channel 224 x 224 image, vector of length num_classes, label)
        return image, multi_hot, binary_label  
    
    def __len__(self):
        """Return sample count"""
        return len(self.samples)
    
    def __repr__(self):
        """Return string representation of dataset"""
        pos_count = sum(1 for _, _, binary_label in self.samples if binary_label == 1)
        neg_count = len(self.samples) - pos_count
        return (f"DvXrayDataset(samples={len(self.samples)}, pos={pos_count}, neg={neg_count}, "
                f"classes={self.num_classes})\n"
                f"Labels: {', '.join(f'{v}:{k}' for k, v in self.label_map.items())}")

### Load the DvXray Dataset

In [9]:
# EfficientNet expects 224x224 images (ImageNet standard)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 224 is the resolution of B0 model
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406]
    , std=[0.229, 0.224, 0.225]), # values are of the ILSVRC-2012 dataset that EfficientNet-B0 was pretrained on.
])

# Load the data (downloads if not present)
dataset = DvXrayDataset(transform=transform, download=True)
print(dataset)

FileURLRetrievalError: Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1NK1DWLMztROwRkJIlYAnexWLgyFv_gDF

but Gdown can't. Please check connections and permissions.

In [ ]:
# Show first 2 classes
visualize_samples(dataset, n_classes=2)

### Split dataset (8-1-1) into Train-Validate-Test

In [ ]:
from torch.utils.data import Subset

# Split the subset
scale = 0.005  # NOTE; set this to 1.0 during actual training 
total_size = int(len(dataset) * scale)
indices = torch.randperm(len(dataset))[:total_size].tolist()
subset = Subset(dataset, indices)

validate_size = int(0.10 * total_size)
test_size = int(0.10 * total_size)
train_size = total_size - validate_size - test_size

train_set, validate_set, test_set = random_split(
    subset, [train_size, validate_size, test_size]
)

batch_size = 32
train_loader = DataLoader(train_set, batch_size, shuffle=True)
val_loader = DataLoader(validate_set, batch_size)
test_loader = DataLoader(test_set, batch_size)

print(f"Train set size: {len(train_set)}")
print(f"Validate set size: {len(validate_set)}")
print(f"Test set size: {len(test_set)}\n")
print(f"Number of classes: {dataset.num_classes}")
print(f"Sample shape: {train_set[0][0].shape}")

### EfficientNet Model
We use a pretrained EfficientNet-B0 from torchvision and adapt it for:
- 6-channel input (dual-view X-ray: OL + SD) by averaging the first conv layer weights
- Two output heads: multi-label classification and binary threat detection

In [ ]:
class MyEfficientNet(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        """Adapt EfficientNet_B0 to the dataset."""
        super().__init__()
        
        # Load pretrained EfficientNet-B0
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.efficientnet = models.efficientnet_b0(weights=weights)
        
        # Modify first conv layer to accept 6-channel input (OL:3 + SD:3) instead of 3 
        old_conv = self.efficientnet.features[0][0]
        new_conv = nn.Conv2d(6, 32, kernel_size=3, stride=2, padding=1, bias=False) # kernel size, stride, padding follow those of EfficientNet
        with torch.no_grad():
            mean_weight = old_conv.weight.mean(dim=1, keepdim=True)         # average the 3-channel weights
            new_conv.weight = nn.Parameter(mean_weight.repeat(1, 6, 1, 1))  # use averaged weight for 6-channel weights 
        self.efficientnet.features[0][0] = new_conv
        
        # Modify last classifier layer to have two output heads: 1280 → 512 → {num_classes, 2}
        in_features = self.efficientnet.classifier[1].in_features  # 1280
        self.efficientnet.classifier = nn.Identity()   # to retain 1280 feature vector (EfficientNet: 1280 -> 1000 classes)
        self.shared_hidden = nn.Linear(in_features, 512)
        self.dropout = nn.Dropout(0.3)                 # tuneable
        self.multi_label_head = nn.Linear(512, num_classes)    # multi-label classifier (16 classes)
        self.binary_head = nn.Linear(512, 2)             # binary classifier 
    
    def forward(self, x):
        """Forward pass. (batch, 6, 224, 224) → multi_logits, binary_logits"""
        x = self.efficientnet(x)
        x = self.shared_hidden(x)
        x = F.relu(x)
        x = self.dropout(x)
        multi_out = self.multi_label_head(x)    # object classification
        binary_out = self.binary_head(x)  # threat/no threat
        
        return multi_out, binary_out

    def freeze_layers(self, *targets):
        """Freeze selected layers of the model.
        
        Args:
            targets: One or more of:
                - 'features' (str): freeze all feature extractor layers (features[0] through features[-1])
                - 'heads' (str): freeze the custom FC heads (shared_hidden, multi_label_head, binary_head)
                - 'initial_conv' (str): freeze only the first conv layer (6-channel adapter)
                - int: freeze a specific features block by index (e.g. 0 = initial conv+bn, 1 = first MBConv block)
                - range/list of ints: freeze multiple features blocks (e.g. range(3) or [0,1,2])
        
        Examples:
            model.freeze_layers('features')           # freeze entire backbone
            model.freeze_layers('heads')              # freeze only custom heads
            model.freeze_layers('initial_conv')       # freeze just the 6-channel adapter
            model.freeze_layers(range(3))             # freeze features[0], [1], [2]
            model.freeze_layers(1, 2, 3)              # freeze features[1], [2], [3]
        """
        # Collect parameter groups to freeze
        params_to_freeze = []
        
        for target in targets:
            if target == 'features':
                params_to_freeze.extend(self.efficientnet.features.parameters())
            elif target == 'heads':
                params_to_freeze.extend(self.shared_hidden.parameters())
                params_to_freeze.extend(self.multi_label_head.parameters())
                params_to_freeze.extend(self.binary_head.parameters())
            elif target == 'initial_conv':
                params_to_freeze.extend(self.efficientnet.features[0].parameters())
            elif isinstance(target, int):
                params_to_freeze.extend(self.efficientnet.features[target].parameters())
            elif isinstance(target, (range, list)):
                for idx in target:
                    params_to_freeze.extend(self.efficientnet.features[idx].parameters())
            else:
                raise ValueError(f"Unknown freeze target: {target}")
        
        for param in params_to_freeze:
            param.requires_grad = False

    def unfreeze_all(self):
        """Unfreeze all layers (make everything trainable)."""
        for param in self.parameters():
            param.requires_grad = True

    def print_trainable_summary(self):
        """Print a summary of which layers are frozen vs trainable."""
        print("\n=== Trainable Layers Summary ===")
        
        # Features
        for i, block in enumerate(self.efficientnet.features):
            trainable = sum(1 for p in block.parameters() if p.requires_grad)
            total = sum(1 for p in block.parameters())
            status = "TRAINABLE" if trainable > 0 else "FROZEN"
            block_name = f"Initial Conv+BN" if i == 0 else f"MBConv Block {i}"
            print(f"  features[{i}] ({block_name:20s}): {status} ({trainable}/{total} params)")
        
        # Heads
        head_descriptions = {
            'shared_hidden': 'Shared Hidden Layer',
            'multi_label_head': 'Multi-Label Head',
            'binary_head': 'Binary Head',
        }
        for name, module in [('shared_hidden', self.shared_hidden), ('multi_label_head', self.multi_label_head), ('binary_head', self.binary_head)]:
            trainable = sum(1 for p in module.parameters() if p.requires_grad)
            total = sum(1 for p in module.parameters())
            status = "TRAINABLE" if trainable > 0 else "FROZEN"
            print(f"  {name:16s} ({head_descriptions[name]:20s}): {status} ({trainable}/{total} params)")
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\n  Total: {trainable_params:,} / {total_params:,} parameters trainable ({100*trainable_params/total_params:.1f}%)\n")

In [ ]:
# Initialize EfficientNet model
model = MyEfficientNet(num_classes=dataset.num_classes, pretrained=True).to(device)
print(model)

### Freezing layers
We experiment with progressive unfreezing from the back to optimize for best performance.

- **Phase 1:** Freeze all backbone MBConv blocks (`features[1]`–`[8]`). Train only initial conv and the classifier heads.
- **Phase 2:** Unfreeze the last 2 MBConv blocks (`features[7]`–`[8]`). Train initial conv + heads + 2 deep layers.
- **Phase 3:** Unfreeze the last 4 MBConv blocks (`features[5]`–`[8]`). Train initial conv + heads + 4 deep layers.

In [ ]:
# --- Layer Freezing (choose what to freeze) ---
# Uncomment one of these before training:

# Phase 1: freeze all MBConv blocks, train initial conv + heads only
model.unfreeze_all()
model.freeze_layers(1, 2, 3, 4, 5, 6, 7, 8)
model.print_trainable_summary()

# Phase 2: unfreeze last 2 MBConv blocks, train initial conv + heads + deep layers 7, 8
# model.unfreeze_all()
# model.freeze_layers(1, 2, 3, 4, 5, 6)
# model.print_trainable_summary()

# Phase 3: unfreeze last 4 MBConv blocks, train initial conv + heads + deep layers 5, 6, 7, 8
# model.unfreeze_all()
# model.freeze_layers(1, 2, 3, 4)
# model.print_trainable_summary()

### Optimizer and loss function

In [ ]:
# Define loss function
orig_dataset = train_set.dataset.dataset  # Navigate through Subset chain
orig_indices = [train_set.dataset.indices[i] for i in train_set.indices]
labels = [orig_dataset.samples[i][2] for i in orig_indices]
neg = labels.count(0)
pos = labels.count(1)
print(f"Class balance: {neg} negatives, {pos} positives")
weights = torch.tensor([1.0, neg / pos]).to(device)
criterion_multi = nn.BCEWithLogitsLoss()                # handles multi-hot targets
criterion_binary = nn.CrossEntropyLoss(weight=weights)  # handles binary 

# Select optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

### Training

In [ ]:
# Train model
best_val_acc = 0    # to identify overfitting 
train_losses, val_losses, val_accs = [], [], []     # for caching
num_epochs = 20
for epoch in tqdm(range(num_epochs), desc="Epochs"):
    
    model.train()

    # Process train batches
    total_loss = 0
    for images, multi_labels, binary_labels in tqdm(train_loader, desc="Train", leave=False):
        # Predict labels
        images = images.to(device)
        multi_out, binary_out = model(images)

        # Calculate loss
        multi_labels = multi_labels.to(device)
        binary_labels = binary_labels.to(device)
        loss_multi = criterion_multi(multi_out, multi_labels)
        loss_binary = criterion_binary(binary_out, binary_labels)
        loss = loss_multi + loss_binary
        total_loss += loss.item()

        # Backpropagate
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Cache losses 
    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)

    # Evaluate model (to find overfitting and save best weights)
    val_loss = 0
    correct = 0
    total = 0    
    with torch.no_grad():
        
        model.eval()

        # Process validate batches
        for images, multi_labels, binary_labels in tqdm(val_loader, desc="Val", leave=False):
            # Predict labels
            images = images.to(device)
            multi_out, binary_out = model(images)
            
            # Calculate loss
            multi_labels = multi_labels.to(device)
            binary_labels = binary_labels.to(device)
            loss_multi = criterion_multi(multi_out, multi_labels)
            loss_binary = criterion_binary(binary_out, binary_labels)
            loss = loss_multi + loss_binary
            val_loss += loss.item()

            # Count accurate binary predictions
            preds = torch.argmax(binary_out, dim=1)
            correct += (preds == binary_labels).sum().item()
            total += binary_labels.size(0)

    # Cache losses and accuracy
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Print epoch training result
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save weights with of best models 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_efficientnet_model.pth')
        print(f"  → Best model saved (val_acc: {val_acc:.4f})")

# Load best weights before testing
model.load_state_dict(torch.load('best_efficientnet_model.pth', weights_only=True))
print(f"\nLoaded best model with val_acc: {best_val_acc:.4f}")

### Evaluation

In [ ]:
def train_model(model, experiment_name: str, override=False):
    """Train model and save weights + history.
    
    This function needs the following objects so make sure they appear in the same scope as the function: 
    - train_loader, 
    - val_loader, 
    - criterion_multi, 
    - criterion_binary, 
    - optimizer 
    
    """
    # Check if results already exists
    save_path = Path("experiments") / experiment_name
    save_path.mkdir(parents=True, exist_ok=True)
    
    results_file = save_path / "results.pkl"
    model_file = save_path / "best_model.pth"
    
    if not override and results_file.exists() and model_file.exists():
        print(f"⚠️ Training already done for {experiment_name}")
        return load_training_results(experiment_name)
    
    # Train
    best_val_acc = 0    # to identify overfitting 
    train_losses, val_losses, val_accs = [], [], []     # for caching
    num_epochs = 20
    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        
        model.train()

        # Process train batches
        total_loss = 0
        for images, multi_labels, binary_labels in tqdm(train_loader, desc="Train", leave=False):
            # Predict labels
            images = images.to(device)
            multi_out, binary_out = model(images)

            # Calculate loss
            multi_labels = multi_labels.to(device)
            binary_labels = binary_labels.to(device)
            loss_multi = criterion_multi(multi_out, multi_labels)
            loss_binary = criterion_binary(binary_out, binary_labels)
            loss = loss_multi + loss_binary
            total_loss += loss.item()

            # Backpropagate
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        # Cache losses 
        train_loss = total_loss / len(train_loader)
        train_losses.append(train_loss)

        # Evaluate model (to find overfitting and save best weights)
        val_loss = 0
        correct = 0
        total = 0    
        with torch.no_grad():
            
            model.eval()

            # Process validate batches
            for images, multi_labels, binary_labels in tqdm(val_loader, desc="Val", leave=False):
                # Predict labels
                images = images.to(device)
                multi_out, binary_out = model(images)
                
                # Calculate loss
                multi_labels = multi_labels.to(device)
                binary_labels = binary_labels.to(device)
                loss_multi = criterion_multi(multi_out, multi_labels)
                loss_binary = criterion_binary(binary_out, binary_labels)
                loss = loss_multi + loss_binary
                val_loss += loss.item()

                # Count accurate binary predictions
                preds = torch.argmax(binary_out, dim=1)
                correct += (preds == binary_labels).sum().item()
                total += binary_labels.size(0)

        # Cache losses and accuracy
        val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Print epoch training result
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Save weights with of best models 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_efficientnet_model.pth')
            print(f"  → Best model saved (val_acc: {val_acc:.4f})")

    # Done with training, save results with pickle
    results = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc,
        'best_epoch': val_accs.index(max(val_accs)) + 1,
        'experiment_name': experiment_name,
        'num_epochs': num_epochs
    }
    
    with open(results_file, 'wb') as f:
        pickle.dump(results, f)
    
    # Model weights already saved during training (when val_acc improved)
    print(f"✅ Training complete: {experiment_name}")
    print(f"   Best val_acc: {best_val_acc:.4f}")
    
    return results


def load_training_results(experiment_name):
    """Load previously saved training results."""
    results_file = Path("experiments") / experiment_name / "results.pkl"
    
    if results_file.exists():
        with open(results_file, 'rb') as f:
            return pickle.load(f)
    else:
        print(f"⚠️ No results found for {experiment_name}")
        return None

In [ ]:
def test_model(model_class, experiment_name, dataset, device):
    """Test a saved model on test set."""
    global test_loader  # Use the same test_loader from your notebook
    
    save_path = Path("experiments") / experiment_name
    model_file = save_path / "best_model.pth"
    results_file = save_path / "results.pkl"
    
    if not model_file.exists():
        print(f"⚠️ No model found for {experiment_name}")
        return None
    
    # Load training results to get best_val_acc
    training_results = load_training_results(experiment_name)
    
    # Create and load model
    model = model_class(num_classes=dataset.num_classes, pretrained=True).to(device)
    model.load_state_dict(torch.load(model_file, weights_only=True))
    model.eval()
    
    # Test
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, _, binary_labels in test_loader:
            images = images.to(device)
            binary_labels = binary_labels.to(device)
            
            _, binary_out = model(images)
            preds = torch.argmax(binary_out, dim=1)
            
            correct += (preds == binary_labels).sum().item()
            total += binary_labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(binary_labels.cpu().numpy())
    
    test_acc = correct / total
    
    # Save test results
    test_results = {
        'test_acc': test_acc,
        'best_val_acc': training_results['best_val_acc'] if training_results else None,
        'experiment_name': experiment_name,
        'total_samples': total,
        'correct': correct
    }
    
    test_file = save_path / "test_results.pkl"
    with open(test_file, 'wb') as f:
        pickle.dump(test_results, f)
    
    print(f"\n{'='*50}")
    print(f"📊 Test Results for {experiment_name}")
    print(f"   Best Val Acc: {test_results['best_val_acc']:.4f}")
    print(f"   Test Acc: {test_acc:.4f} ({correct}/{total})")
    print(f"{'='*50}\n")
    
    # Optional: print classification report
    from sklearn.metrics import classification_report
    print(classification_report(all_labels, all_preds, target_names=['No Threat', 'Threat']))
    
    return test_results

In [ ]:
# Phase 1: Train only
model1 = MyEfficientNet(num_classes=dataset.num_classes, pretrained=True).to(device)
model1.freeze_layers(1, 2, 3, 4, 5, 6, 7, 8)
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-4)

results1 = train_model(model1, experiment_name="phase1_frozen_all", override=False)

# Phase 2: Train only
model2 = MyEfficientNet(num_classes=dataset.num_classes, pretrained=True).to(device)
model2.freeze_layers(1, 2, 3, 4, 5, 6)
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-4)

results2 = train_model(model2, experiment_name="phase2_unfrozen_7_8", override=False)

# Later (or in a separate notebook), test all saved models
test1 = test_model(MyEfficientNet, "phase1_frozen_all", dataset, device)
test2 = test_model(MyEfficientNet, "phase2_unfrozen_7_8", dataset, device)

# Compare
print("\n📊 Final Comparison:")
print(f"Phase 1: Val={test1['best_val_acc']:.4f}, Test={test1['test_acc']:.4f}")
print(f"Phase 2: Val={test2['best_val_acc']:.4f}, Test={test2['test_acc']:.4f}")

In [ ]:
# ================= FINAL TEST =================
model.eval()
correct = 0
total = 0
printed = False
# Multi-class names for report
idx_to_label = {v: k for k, v in dataset.label_map.items()}


all_preds = []
all_labels = []

with torch.no_grad():
    for images, _, binary_labels in test_loader:
        images = images.to(device)
        binary_labels = binary_labels.to(device)

        multi_out, binary_out = model(images)
        preds = torch.argmax(binary_out, dim=1)

        correct += (preds == binary_labels).sum().item()
        total += binary_labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(binary_labels.cpu().numpy())
    
        if not printed:
            # Show multi-label predictions (top predicted classes)
            probs = torch.sigmoid(multi_out)
            topk = torch.topk(probs[0], k=3)
            top_labels = [idx_to_label[i.item()] for i in topk.indices]
            top_probs = [f"{p:.3f}" for p in topk.values]
            print(f"Top predictions: {list(zip(top_labels, top_probs))}")

            pred_binary = torch.argmax(binary_out, dim=1)
            print("Pred threat:", ["Threat" if i.item()==1 else "No Threat" for i in pred_binary[:5]])
            printed = True
    
print("\n" + "="*50)
print(classification_report(all_labels, all_preds, target_names=['No Threat', 'Threat']))
print("Final Test Accuracy:", correct / total)

### Training Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, num_epochs+1), train_losses, label='Train Loss')
axes[0].plot(range(1, num_epochs+1), val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()

axes[1].plot(range(1, num_epochs+1), val_accs, label='Val Accuracy', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
the 
plt.tight_layout()
plt.savefig('efficientnet_training_curves.png', dpi=150)
plt.show()